# Airbourne Song Classifier 🎸 (a joke, mostly)

A small neural net that listens to a song and decides whether it's:

- an **Airbourne** song with **meaningful** lyrics,
- an **Airbourne** song with **meaningless** lyrics (statistically, most of them), or
- **not Airbourne** at all.

Runs end-to-end on the free Google Colab **T4** GPU (Runtime → Change runtime type → T4 GPU).
This one notebook builds the dataset (from YouTube links), trains, and runs one-off
inference (also from a YouTube link).

**Model persistence, in order:**
1. Cloning/pulling this repo pulls the latest **git-lfs** committed checkpoint — the durable baseline.
2. If you opt into `USE_DRIVE_CACHE` and a checkpoint exists in your **Google Drive** cache, it
   overrides that baseline (Drive is your fast, no-commit-needed working copy).
3. Training automatically saves a copy to the Drive cache too, if enabled.
4. Publishing a new checkpoint back to GitHub via git-lfs isn't part of this notebook — do it
   from a terminal (`git lfs track`, `git add`, `git commit`, `git push`) when you want to.

**Caveat, for honesty's sake:** this model only hears audio, not lyrics — "meaningful vs.
meaningless" is really a per-song acoustic fingerprint it's memorizing, not an assessment of
Joel O'Keeffe's poetry. Don't commit raw mp3s to the repo; `data/raw/` is gitignored on purpose.

## 1. Setup

In [ ]:
import os
from pathlib import Path

REPO_HTTPS_URL = "https://github.com/vpbukhti/airborne-classifier.git"
REPO_DIR = Path("/content/airborne-classifier")

# git-lfs must be installed *before* cloning/pulling, so the checkpoint's real bytes
# get smudged in on checkout instead of leaving a small LFS pointer file behind.
!apt-get -qq update && apt-get -qq install -y git-lfs
!git lfs install

if not REPO_DIR.is_dir():
    !git clone {REPO_HTTPS_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    !git -C {REPO_DIR} lfs pull

os.chdir(REPO_DIR)
print("working dir:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# Not `pip install -e .`: an editable install's .pth redirector is only picked up
# by a *fresh* interpreter's site init, but this pip install and this import share
# the same already-running kernel. Adding src/ to sys.path works immediately instead.
import sys

sys.path.insert(0, str(REPO_DIR / "src"))

import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch

import airbourne_classifier as ac
from airbourne_classifier import storage
from airbourne_classifier.dataset import scan_dataset

print("CUDA available:", torch.cuda.is_available())

## 2. Config — your choices

In [ ]:
# --- your choices --------------------------------------------------------
USE_DRIVE_CACHE = False   # True: cache both the model AND your YouTube-built dataset on Drive
FORCE_RETRAIN   = False   # True: retrain even if a checkpoint was already loaded
# ---------------------------------------------------------------------------

DATA_DIR   = REPO_DIR / "data" / "raw"                  # data/raw/{meaningful,meaningless,non_airbourne}/*.mp3
MODEL_PATH = REPO_DIR / "models" / "airbourne_classifier.pt"

DRIVE_CACHE_DIR  = Path("/content/drive/MyDrive/airborne-classifier-cache")
DRIVE_MODEL_PATH = DRIVE_CACHE_DIR / "airbourne_classifier.pt"
DRIVE_DATA_DIR   = DRIVE_CACHE_DIR / "data" / "raw"

EPOCHS = 15
BATCH_SIZE = 16

## 3. Google Drive (optional)

If `USE_DRIVE_CACHE` is on: mounts Drive, restores any dataset you've built in previous
sessions, and restores a cached model checkpoint if one's there (overriding the git-lfs
baseline from Setup).

In [ ]:
if USE_DRIVE_CACHE:
    storage.mount_drive()

    if DRIVE_DATA_DIR.is_dir():
        shutil.copytree(DRIVE_DATA_DIR, DATA_DIR, dirs_exist_ok=True)
        n_files = sum(1 for p in DATA_DIR.rglob("*") if p.is_file())
        print(f"restored {n_files} dataset file(s) from Drive")

    model_from_drive = storage.restore_from_drive(DRIVE_MODEL_PATH, MODEL_PATH)
    print("restored model from Drive cache:", model_from_drive)
else:
    model_from_drive = False

## 4. Build your dataset from YouTube (optional)

Pick a label, paste a YouTube link, run the cell. Repeat for every song you want to add
— N Airbourne songs split across `meaningful`/`meaningless`, plus M songs that aren't
Airbourne. Files are named by YouTube video id, so re-running with the same link just
overwrites in place rather than duplicating. Downloads go to `data/raw/<label>/` and are
mirrored to your Drive cache automatically if `USE_DRIVE_CACHE` is on.

In [ ]:
#@title Add a song { run: "auto" }
LABEL = "meaningful" #@param ["meaningful", "meaningless", "non_airbourne"]
YOUTUBE_URL = "" #@param {type:"string"}

if YOUTUBE_URL.strip():
    path = ac.add_labeled_song(
        YOUTUBE_URL.strip(),
        LABEL,
        data_dir=DATA_DIR,
        drive_dir=(DRIVE_DATA_DIR if USE_DRIVE_CACHE else None),
    )
    print(f"saved [{LABEL}]: {path}")
else:
    print("paste a YouTube URL above, then re-run this cell")

## 5. Check what you've got

In [ ]:
songs = scan_dataset(DATA_DIR)
counts = Counter(s.label for s in songs)
for label in ac.LABELS:
    print(f"{label:14s} {counts.get(label, 0)}")

## 6. Load a cached model, or train a new one

In [ ]:
have_model = model_from_drive or MODEL_PATH.is_file()

if have_model and not FORCE_RETRAIN:
    print(f"using existing checkpoint at {MODEL_PATH} (set FORCE_RETRAIN=True above to retrain)")
else:
    result = ac.train(
        data_dir=DATA_DIR,
        model_out=MODEL_PATH,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
    )

    epochs_ran = [h["epoch"] for h in result.history]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs_ran, [h["train_loss"] for h in result.history], label="train")
    axes[0].plot(epochs_ran, [h["val_loss"] for h in result.history], label="val")
    axes[0].set_title("loss"); axes[0].legend()
    axes[1].plot(epochs_ran, [h["train_acc"] for h in result.history], label="train")
    axes[1].plot(epochs_ran, [h["val_acc"] for h in result.history], label="val")
    axes[1].set_title("accuracy"); axes[1].legend()
    plt.show()

    print("val metrics:", result.val_metrics)
    print("test metrics:", result.test_metrics)

    if USE_DRIVE_CACHE:
        storage.cache_to_drive(DRIVE_MODEL_PATH, MODEL_PATH)
        print("auto-cached the newly trained model to Drive:", DRIVE_MODEL_PATH)

model, audio_config, device = ac.load_model(MODEL_PATH)
print("model loaded, ready for inference")

## 7. Inference: classify any song by YouTube link

In [ ]:
#@title Classify a song { run: "auto" }
INFERENCE_URL = "" #@param {type:"string"}

# This cell has run: "auto", so editing the field above re-runs just this cell —
# make sure a model is actually loaded even if section 6 hasn't run yet this session
# (e.g. right after a runtime restart).
if "model" not in globals():
    model, audio_config, device = ac.load_model(MODEL_PATH)
    print("(re)loaded model from", MODEL_PATH)

if INFERENCE_URL.strip():
    prediction = ac.predict_youtube(model, audio_config, INFERENCE_URL.strip(), device=device)
    print(f"predicted: {prediction['label']}")
    for label, prob in prediction["probabilities"].items():
        print(f"  {label:14s} {prob:.3f}")
else:
    print("paste a YouTube URL above, then re-run this cell")